# VisionReasoner: Prompt Learning & RL
Este notebook utiliza a abordagem de **Prompt Learning** baseado no feedback do usuário (IoU/Reward). 
A ideia é que as queries (prompts) não sejam estáticas. O modelo testa diferentes prompts, você avalia usando a ferramenta `anotador_rl.py`, e o notebook **evolui e descobre os melhores prompts**.

In [2]:
import sys
!{sys.executable} -m pip install transformers sam2 openai ultralytics

In [3]:
import os
import sys
import json
import random
import re
import numpy as np
from tqdm import tqdm

# Adiciona o caminho do VisionReasoner
sys.path.append('/media/workstation_2x_3080/HD 1TB/Famez_yolo/VisionReasoner')

# Importe o seu modelo (ajuste conforme a sua API)
from vision_reasoner.models.vision_reasoner_model import VisionReasonerModel

/home/workstation_2x_3080/miniconda3/envs/yolo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# ==========================================
# 1. INICIALIZAÇÃO DO MODELO E CONFIGS
# ==========================================
ARQUIVO_FEEDBACK = 'rl_feedback.json'
ARQUIVO_PROMPTS = 'prompts_pool.json'

caminhos_fotos = [
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Foto colorimetro-Julho2026',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Fotos colorímetro- fevereiro-2026',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/fotos colorímetro-dezembro-2025'
]

print("Carregando VisionReasoner...")
model = VisionReasonerModel(reasoning_model_path='/media/workstation_2x_3080/HD 1TB/Famez_yolo/VisionReasoner/pretrained_models/VisionReasoner-7B')
print("Modelo carregado!")

Carregando VisionReasoner...
FlashAttention2 not available for Reasoning Model, falling back to SDPA...


Loading weights: 100%|██████████| 729/729 [00:02<00:00, 269.13it/s]


FlashAttention2 not available for TaskRouter, falling back to SDPA...


Loading weights: 100%|██████████| 339/339 [00:00<00:00, 578.00it/s]


Modelo carregado!


In [11]:
# ==========================================
# 2. GERENCIAMENTO DO POOL DE PROMPTS (RL)
# ==========================================
def carregar_prompts():
    if os.path.exists(ARQUIVO_PROMPTS):
        with open(ARQUIVO_PROMPTS, 'r') as f:
            return json.load(f)
    
    # Prompts iniciais (Seed)
    return {
        "p1": {
            "texto": "Analyze this image of cattle skin inside a circular PVC pipe. Find the precise bounding box of the exposed dark skin patch in the center. Ignore white labels. Return the coordinates as [x1, y1, x2, y2].",
            "recompensas": []
        },
        "p2": {
            "texto": "Identify the central dark spot (hair or skin) within the circular area. Disregard any alphanumeric tags. Output bounding box format: [x1, y1, x2, y2].",
            "recompensas": []
        },
        "p3": {
            "texto": "Find the dark skin lesion or hair sample inside the PVC pipe hole. Provide the bounding box coordinates [x1, y1, x2, y2] strictly for the biological sample, ignoring the white background and text.",
            "recompensas": []
        },
        "p4": {
            "texto": "Detect the central biological specimen (dark skin/hair). Focus only on the circular opening. Do not detect the PVC pipe or the printed text labels. Format: [x1, y1, x2, y2].",
            "recompensas": []
        },
        "p5": {
            "texto": "Locate the dark region of interest in the center of the image. The object is a cattle skin patch. Return its bounding box [x1, y1, x2, y2].",
            "recompensas": []
        }
    }

def salvar_prompts(pool):
    with open(ARQUIVO_PROMPTS, 'w') as f:
        json.dump(pool, f, indent=4)



In [12]:
# ==========================================
# 3. EXTRAÇÃO VIA PROMPT LEARNING
# ==========================================
def carregar_feedback():
    if os.path.exists(ARQUIVO_FEEDBACK):
        with open(ARQUIVO_FEEDBACK, 'r') as f:
            return json.load(f)
    return {}

def salvar_feedback(dados):
    with open(ARQUIVO_FEEDBACK, 'w') as f:
        json.dump(dados, f, indent=4)

def pedir_bbox_vlm(image_path, prompt_text):
    from PIL import Image
    import re
    image = Image.open(image_path).convert('RGB')
    result = model.detect_objects(image, prompt_text)
    if result and 'bboxes' in result and len(result['bboxes']) > 0:
        return tuple(result['bboxes'][0])
    
    # Fallback:
    resposta_texto = result.get('full_response', '')
    try:
        numeros = re.findall(r'\d+', resposta_texto)
        if len(numeros) >= 4:
            return tuple(map(int, numeros[:4]))
    except Exception as e:
        print(f"Falha ao parsear resposta: {resposta_texto}")
    return None

def gerar_prompt_adaptativo(image_path, historico_pool):
    from PIL import Image
    import numpy as np
    image = Image.open(image_path).convert('RGB')
    
    boas_dicas = []
    for pid, data in historico_pool.items():
        if data['recompensas']:
            media = np.mean(data['recompensas'])
            if media > 0.5:
                boas_dicas.append(data['texto'])
                
    dicas_str = "\n- ".join(boas_dicas[-3:])
    if not dicas_str:
        dicas_str = "Find the dark skin patch in the center."
        
    meta_prompt = f"""You are an AI optimization assistant.
Analyze this image of a cattle skin patch inside a PVC pipe.
We want to detect the central dark biological sample (skin/hair).
Here are some queries that worked well recently:
- {dicas_str}

Based on this image and the successful history, generate a NEW, perfectly adapted object detection query to find the target in THIS specific image. Return ONLY the new query text."""

    result = model.answer_question(image, meta_prompt)
    novo_prompt = result.get('answer', dicas_str.split('\n')[0].replace('- ', '')).strip()
    novo_prompt = novo_prompt.replace('"', '').replace("'", "")
    return novo_prompt

prompts_pool = carregar_prompts()
feedback = carregar_feedback()

todas_as_fotos = []
for pasta in caminhos_fotos:
    if not os.path.exists(pasta): continue
    for f in os.listdir(pasta):
        if f.lower().endswith(('.jpg', '.png')):
            todas_as_fotos.append(os.path.join(pasta, f))

print(f"Testando predições em {len(todas_as_fotos)} imagens...")

for caminho_img in tqdm(todas_as_fotos):
    nome_arquivo = os.path.basename(caminho_img)
    if nome_arquivo in feedback: continue
        
    prompt_dinamico = gerar_prompt_adaptativo(caminho_img, prompts_pool)
    
    pid = f"dinamico_{len(prompts_pool) + 1}"
    prompts_pool[pid] = {'texto': prompt_dinamico, 'recompensas': []}
    salvar_prompts(prompts_pool)
    
    bbox_predito = pedir_bbox_vlm(caminho_img, prompt_dinamico)
    
    if bbox_predito:
        feedback[nome_arquivo] = {
            'auto_coords': bbox_predito,
            'prompt_id': pid,
            'prompt_text': prompt_dinamico,
            'status_auto': 'VISION_REASONER_RL',
            'reward': None
        }
        salvar_feedback(feedback)
        
print("Previsões do Lote finalizadas. Use o anotador_rl.py agora!")

Testando predições em 266 imagens...


100%|██████████| 266/266 [00:00<00:00, 965960.92it/s]

Previsões do Lote finalizadas. Use o anotador_rl.py agora!


In [14]:
# ==========================================
# 4. ATUALIZAR REWARDS E MUTAR PROMPTS
# ==========================================
# Rode esta célula DEPOIS de anotar as imagens na ferramenta visual

feedback = carregar_feedback()
prompts_pool = carregar_prompts()

# 4.1 Extrai os rewards dados por você e injeta no pool
for arquivo, dados in feedback.items():
    if dados.get('reward') is not None and 'prompt_id' in dados:
        pid = dados['prompt_id']
        if pid in prompts_pool:
            prompts_pool[pid]['recompensas'].append(dados['reward'])
            del dados['prompt_id'] 

salvar_feedback(feedback) 

# 4.2 Mostra o Placar
print("\n--- PLACAR DOS PROMPTS ---")
medias = {}
for pid, data in prompts_pool.items():
    if data['recompensas']:
        media = np.mean(data['recompensas'])
        medias[pid] = media
        print(f"{pid} | Reward Médio: {media:.3f} | Text: {data['texto'][:50]}...")

# 4.3 Evolução (Mutation) Automática
if medias:
    melhor_pid = max(medias, key=medias.get)
    melhor_prompt = prompts_pool[melhor_pid]['texto']
    
    # Gera um ID para o novo prompt
    novo_pid = f"p{len(prompts_pool) + 1}"
    
    print(f"\nO melhor prompt atual é o {melhor_pid}.")
    print(f"Gerando mutação automática para criar o {novo_pid}...")
    
    meta_prompt = (
        "You are an AI optimization assistant. The following prompt is used to command a Vision-Language Model "
        "to detect the bounding box of a cattle skin patch inside a PVC pipe. \n"
        f"Current Prompt: '{melhor_prompt}'\n"
        "Rewrite and improve this prompt to be more precise, robust, and focus strictly on finding the dark "
        "skin/hair region while ignoring white alphanumeric labels. Output ONLY the new prompt text."
    )
    
    try:
        # Usa o próprio VisionReasoner para fazer o papel de LLM textual.
        # Passamos a primeira imagem do dataset só para preencher o argumento obrigatório da função.
        caminho_amostra = os.path.join(caminhos_fotos[0], os.listdir(caminhos_fotos[0])[0])
        from PIL import Image
        img_obj = Image.open(caminho_amostra).convert('RGB')
        res = model.answer_question(img_obj, meta_prompt)
        novo_texto = res.get('answer', '')
        
        # Limpa o texto gerado de possíveis aspas
        novo_texto = novo_texto.replace('"', '').replace("'", "").strip()
        
        prompts_pool[novo_pid] = {
            "texto": novo_texto,
            "recompensas": []
        }
        
        print(f"Mutação concluída!\nNovo prompt ({novo_pid}): {novo_texto}")
    except Exception as e:
        print("Erro ao tentar mutar o prompt automaticamente:", e)

salvar_prompts(prompts_pool)



--- PLACAR DOS PROMPTS ---
dinamico_6 | Reward Médio: 0.000 | Text: find the dark skin patch in the center of the circ...
dinamico_9 | Reward Médio: 0.272 | Text: find the dark central skin patch....
dinamico_10 | Reward Médio: 0.000 | Text: find the dark skin patch in the center of the circ...
dinamico_14 | Reward Médio: 0.196 | Text: find the dark central skin patch....
dinamico_15 | Reward Médio: 0.238 | Text: find the dark skin patch in the center of the oval...
dinamico_19 | Reward Médio: 0.804 | Text: find the dark central skin patch....
dinamico_21 | Reward Médio: 0.000 | Text: find the dark central skin patch....
dinamico_23 | Reward Médio: 0.103 | Text: find the dark central skin patch....
dinamico_25 | Reward Médio: 0.428 | Text: find the dark central skin patch....
dinamico_26 | Reward Médio: 0.437 | Text: find the dark central skin patch....
dinamico_32 | Reward Médio: 0.000 | Text: find the dark central skin patch....
dinamico_34 | Reward Médio: 0.032 | Text: find the dar